## Problem 1: Visualise data in an interactive map (10 points)


Create an *interactive* **thematic map** using the skills you learnt during lesson 5. You can use *Folium*,<br>
or any other suitable Python package - feel free to experiment).

The map should contain more than one layer (from at least two different data sets). Pay attention to <br>
classification and visualisation (appropriate colour scheme choice, etc.). Write your code in this <br>
notebook, and store the resulting maps in `.html` format (plus possibly additional files, depending on<br>
your approach) in the `NOTEBOOK_DIRECTORY / "docs"` folder.

### Topic of the map

- Use any of the data sets we have used during the course (travel time data, population grid), or <br>
  any other data set of your interest (for example, [open data sets from Statistics Finland](https://www.stat.fi/org/avoindata/index_en.html), <br>
  or the open geo-data from [the Helsinki Region Infoshare](https://hri.fi/en)). If you download <br>
  data sets, add, commit, and push them to the `DATA_DIRECTORY`. Be sure to include an attribution <br>
  to each data set in your output map!
- Feel free to start from and adapt examples from the course material! You can take one step further <br>
  and add additional analysis to, e.g., the shopping centre dominance or the population within vincinity <br>
  of certain addresses)
- This should **not** be an interactive version of your submission for *problem 1*! Create something new!

### Criteria

- The map has more than one data layer (for instance, the results of an analysis, and the road network <br>
  to illustrate a complex spatial pattern). If (and only if) you think it adds additional value, feel <br>
  free to add a base map (but a base map does not count as an own layer) (2 points)
- The map portrays the output of an analysis or a classification that adds value beyond just displaying <br>
  the raw data (2 points)
- The map follows good cartographic practices (appropriate map extent and zoom level, colour scheme, <br>
  legend and data sources, etc.) (3 points)
- The map demonstrates the skills learnt during lesson 5, such as interactive elements (popups, etc.). (3 points)

### Output

- Save the map(s) in html format in the `NOTEBOOK_DIRECTORY / "docs"` folder, remember to add, commit, <br>
  and push them to the GitHub repository
- Remember to also add, commit, and push input data sets and code

---

## 1. Setup

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
import os
import warnings
warnings.filterwarnings('ignore')

print(f"Folium version: {folium.__version__}")

In [ ]:
# File paths
BASE_PATH = "/workspaces/rural-migration-land-use-spain/data"

SPATIAL_DATA = f"{BASE_PATH}/spatial/derived/mun_geographic_administrative_hierarchy.gpkg"
DENSITY_DATA = f"{BASE_PATH}/demography/derived/demography_population_density_1996_2024.csv"
POPULATION_DATA = f"{BASE_PATH}/demography/processed/01_padron_clean_1996_2024.csv"
OUTPUT_DIR = f"{BASE_PATH}/maps"

SELECTED_YEARS = [1996, 2010, 2024]

print(f"Years: {SELECTED_YEARS}")

## 2. Load Data

In [ ]:
# Load geographic boundaries
gdf = gpd.read_file(SPATIAL_DATA)
gdf = gdf[['Mun_Code', 'Mun_Name', 'Comarca_Name', 'Prov_Name', 'geometry']].copy()
gdf.columns = ['Mun_Code', 'Mun', 'Comarca', 'Province', 'geometry']

# Ensure Mun_Code is string with 5 digits
gdf['Mun_Code'] = gdf['Mun_Code'].astype(str).str.zfill(5)

# Convert to WGS84 for Folium
if gdf.crs != 'EPSG:4326':
    gdf = gdf.to_crs(epsg=4326)

# Light simplification (tolerance ~50m) - remove this line if you want full detail
gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.0005, preserve_topology=True)

print(f"Municipalities in gpkg: {len(gdf)}")
print(f"Sample Mun_Code: {gdf['Mun_Code'].iloc[0]}")

In [ ]:
# Load density data
density_df = pd.read_csv(DENSITY_DATA)

# Ensure Mun_Code is string with 5 digits (CRITICAL for matching)
density_df['Mun_Code'] = density_df['Mun_Code'].astype(str).str.zfill(5)

# Filter selected years
density_df = density_df[density_df['Year'].isin(SELECTED_YEARS)].copy()

# Drop Mun column if exists (we'll use the one from gpkg)
if 'Mun' in density_df.columns:
    density_df = density_df.drop(columns=['Mun'])

print(f"Density records: {len(density_df)}")
print(f"Sample Mun_Code: {density_df['Mun_Code'].iloc[0]}")
print(f"Columns: {list(density_df.columns)}")

In [ ]:
# Load population data
pop_df = pd.read_csv(POPULATION_DATA)
pop_df = pop_df[pop_df['Cat'] == 'Total'][['Mun_Code', 'Year', 'Pop']].copy()

# Ensure Mun_Code is string with 5 digits
pop_df['Mun_Code'] = pop_df['Mun_Code'].astype(str).str.zfill(5)

print(f"Population records: {len(pop_df)}")

In [ ]:
# Add Pop to density_df
# Create a lookup dictionary for faster assignment
pop_lookup = pop_df.set_index(['Mun_Code', 'Year'])['Pop'].to_dict()

# Add Pop column
density_df['Pop'] = density_df.apply(
    lambda row: pop_lookup.get((row['Mun_Code'], row['Year']), np.nan), 
    axis=1
)

print(f"Pop added. Shape: {density_df.shape}")
print(f"Columns: {list(density_df.columns)}")

In [ ]:
# Check overlap between gpkg and density data
gpkg_codes = set(gdf['Mun_Code'].unique())
density_codes = set(density_df['Mun_Code'].unique())

overlap = gpkg_codes & density_codes
only_gpkg = gpkg_codes - density_codes
only_density = density_codes - gpkg_codes

print(f"Codes in gpkg: {len(gpkg_codes)}")
print(f"Codes in density: {len(density_codes)}")
print(f"Overlap: {len(overlap)}")
print(f"Only in gpkg: {len(only_gpkg)}")
print(f"Only in density: {len(only_density)}")

In [ ]:
# Merge density data with geometry
# Using pd.merge to have full control
merged_gdf = pd.merge(
    density_df,
    gdf[['Mun_Code', 'Mun', 'Comarca', 'Province', 'geometry']],
    on='Mun_Code',
    how='inner'
)

# Convert to GeoDataFrame
merged_gdf = gpd.GeoDataFrame(merged_gdf, geometry='geometry', crs='EPSG:4326')

# Fill missing and create display columns
merged_gdf['Pop'] = merged_gdf['Pop'].fillna(0).astype(int)
merged_gdf['Density_Display'] = merged_gdf['Pop_Density'].round(1)
merged_gdf['Area_Display'] = merged_gdf['Area_km2'].round(1)

print(f"Final records: {len(merged_gdf)}")
print(f"Unique municipalities: {merged_gdf['Mun_Code'].nunique()}")
print(f"Columns: {list(merged_gdf.columns)}")

## 3. Color Scale

In [ ]:
BREAKS = [0, 10, 25, 50, 100, 500, 1000, 5000, 25000]
COLORS = ['#FFFFB2', '#FECC5C', '#FD8D3C', '#F03B20', '#BD0026', '#800026', '#4D0014', '#1A0007']

def get_color(density):
    if pd.isna(density):
        return '#D9D9D9'
    for i, threshold in enumerate(BREAKS[1:]):
        if density < threshold:
            return COLORS[i]
    return COLORS[-1]

colormap = cm.LinearColormap(
    colors=COLORS[:5],
    vmin=0,
    vmax=500,
    caption='Densidad (hab/km²)'
)

print("Color scale defined")

## 4. Create Map

In [ ]:
# Create base map
m = folium.Map(
    location=[40.0, -3.7],
    zoom_start=6,
    tiles='cartodbpositron',
    prefer_canvas=True
)

# Style functions
def style_function(feature):
    return {
        'fillColor': get_color(feature['properties']['Pop_Density']),
        'color': 'white',
        'weight': 0.5,
        'fillOpacity': 0.7
    }

def highlight_function(feature):
    return {
        'fillColor': get_color(feature['properties']['Pop_Density']),
        'color': 'black',
        'weight': 2,
        'fillOpacity': 0.9
    }

# Add layer for each year
for year in SELECTED_YEARS:
    year_gdf = merged_gdf[merged_gdf['Year'] == year].copy()
    show_layer = (year == SELECTED_YEARS[-1])  # Show only last year by default
    
    fg = folium.FeatureGroup(name=str(year), show=show_layer)
    
    folium.GeoJson(
        year_gdf,
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(
            fields=['Mun', 'Province', 'Comarca', 'Density_Display', 'Pop', 'Area_Display'],
            aliases=['Municipio:', 'Provincia:', 'Comarca:', 'Densidad (hab/km²):', 'Población:', 'Área (km²):'],
            localize=True
        )
    ).add_to(fg)
    
    fg.add_to(m)
    print(f"  {year}: {len(year_gdf)} municipios")

# Add legend and layer control
colormap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

print("\nMap created!")

In [ ]:
# Save map
output_path = f"{OUTPUT_DIR}/interactive_density_folium.html"
m.save(output_path)

size_mb = os.path.getsize(output_path) / 1e6
print(f"Saved: {output_path}")
print(f"Size: {size_mb:.1f} MB")

In [ ]:
# Display map
m

## 5. Summary

In [ ]:
summary = merged_gdf.groupby('Year')['Pop_Density'].agg(['min', 'mean', 'median', 'max']).round(1)
print("Density by Year (hab/km²)")
print(summary)